In [3]:

# DATA WRANGLING ON REAL ESTATE MARKET

import pandas as pd
import numpy as np

# 1. Import dataset
df = pd.read_csv("RealEstate_Prices.csv")

print("Original Data:")
print(df.head())

# Clean column names
df.columns = (df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
              .str.replace(r"[^a-z0-9_]", "", regex=True))

print("\nCleaned Column Names:")
print(df.columns.tolist())

# 2. Handle missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Fill missing numerical values with median
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().any() and not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# 3. Merge with another dataset (optional)
# Uncomment if you have a second dataset:
# extra = pd.read_csv("Neighborhood_Data.csv")
# df = pd.merge(df, extra, on="neighborhood", how="left")

# 4. Filter and subset the data
print("\nFiltered Data:")
if "price" in df.columns:
    filtered_df = df[df["price"] > 0]
    print(filtered_df.head())
else:
    print("Price column not found. Available columns:", df.columns.tolist())
    filtered_df = df.copy()

# 5. Encode categorical variables
categorical_cols = filtered_df.select_dtypes(
    include="object"
).columns.tolist()

encoded_df = pd.get_dummies(
    filtered_df, columns=categorical_cols, dtype=int
)

print("\nEncoded Data:")
print(encoded_df.head())

# 6. Aggregate data by neighborhood
if "neighborhood" in filtered_df.columns and "price" in filtered_df.columns:
    print("\nAverage Price by Neighborhood:")
    print(filtered_df.groupby("neighborhood")["price"].mean())

elif "property_type" in filtered_df.columns and "price" in filtered_df.columns:
    print("\nAverage Price by Property Type:")
    print(filtered_df.groupby("property_type")["price"].mean())

else:
    print("\nNeighborhood/property_type or price column not found.")

# 7. Handle outliers using the IQR method
numeric_cols = filtered_df.select_dtypes(include="number").columns

for col in numeric_cols:
    Q1 = filtered_df[col].quantile(0.25)
    Q3 = filtered_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    filtered_df[col] = filtered_df[col].clip(
        lower=lower, upper=upper
    )

print("\nData after handling outliers:")
print(filtered_df.head())

# Save cleaned dataset
filtered_df.to_csv("Cleaned_RealEstate_Prices.csv", index=False)

print("\nData wrangling completed successfully!")
print("Cleaned dataset saved.")


Original Data:
           id             date     price  bedrooms  bathrooms  sqft_living  \
0  7129300520  20141013T000000  221900.0         3       1.00         1180   
1  6414100192  20141209T000000  538000.0         3       2.25         2570   
2  5631500400  20150225T000000  180000.0         2       1.00          770   
3  2487200875  20141209T000000  604000.0         4       3.00         1960   
4  1954400510  20150218T000000  510000.0         3       2.00         1680   

   sqft_lot  floors  waterfront  view  ...  grade  sqft_above  sqft_basement  \
0      5650     1.0           0     0  ...      7        1180              0   
1      7242     2.0           0     0  ...      7        2170            400   
2     10000     1.0           0     0  ...      6         770              0   
3      5000     1.0           0     0  ...      7        1050            910   
4      8080     1.0           0     0  ...      8        1680              0   

   yr_built  yr_renovated  zipcode 